# 模拟电商订单分析

使用六月、七月的模拟订单，逐步从月度销售额定位发生变化的渠道和商品。金额单位为元。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 数据和清洗

In [ ]:
from io import StringIO

订单文本 = """订单编号,下单日期,商品编号,数量,渠道,客户类型,实付金额
001,2025-06-03,P01,2,官网,老客户,200
002,2025-06-05,P02,1,官网,新客户,150
003,2025-06-08,P03,1,平台,新客户,300
004,2025-06-10,P01,3,官网,老客户,300
005,2025-06-12,P02,2,平台,老客户,300
006,2025-06-15,P03,2,官网,新客户,600
007,2025-06-18,P01,1,官网,新客户,100
008,2025-06-20,P02,1,平台,新客户,150
009,2025-06-23,P03,1,官网,老客户,300
010,2025-06-27,P01,2,平台,老客户,200
011,2025-07-02,P01,1,官网,新客户,100
012,2025-07-05,P02,1,官网 ,老客户,150
013,2025-07-08,P03,1,平台,新客户,300
014,2025-07-12,P01,2,平台,老客户,200
015,2025-07-16,P02,2,官网,老客户,300
016,2025-07-20,P03,1,平台,新客户,300
017,2025-07-24,P01,1,官网,新客户,
018,日期未知,P02,1,平台,老客户,150
013,2025-07-08,P03,1,平台,新客户,300
"""

raw = pd.read_csv(
    StringIO(订单文本),
    dtype={
        "订单编号": "str",
        "商品编号": "str"
    }
)

In [ ]:
raw.head()

In [ ]:
重复记录 = raw.loc[raw.duplicated(keep=False)]
重复记录

In [ ]:
订单编号重复记录 = raw.loc[raw.duplicated(subset=["订单编号"],keep=False)]
订单编号重复记录

In [ ]:
clean = raw.drop_duplicates().copy()

In [ ]:
clean["下单时间"] = pd.to_datetime(
    clean["下单日期"],
    format ="%Y-%m-%d",
    errors ="coerce"
)

In [ ]:
clean.loc[clean["下单时间"].isna(),
    ["订单编号","下单日期","下单时间"]]

In [ ]:
clean.loc[clean["实付金额"].isna(),
    ["订单编号","商品编号","数量","实付金额"]]

In [ ]:
clean["渠道"].unique().tolist()

下列两条更正来自这组练习给定的补查信息。

In [ ]:
clean.loc[
    clean["订单编号"] == "017", "实付金额"
] = 100

clean.loc[
    clean["订单编号"] == "018", "下单时间"
] = pd.to_datetime("2025-07-28")

In [ ]:
分析表 = clean.loc[clean["下单时间"].notna() & clean["实付金额"].notna()].copy()

In [ ]:
分析表["月份"] = 分析表["下单时间"].dt.to_period("M")

## 月度与渠道对比

In [ ]:
月报 = 分析表.groupby("月份").agg(纳入订单数=("订单编号","nunique"),
                                  已知金额合计=("实付金额","sum"),
                           平均订单金额=("实付金额","mean"))
月报

In [ ]:
渠道月报 = 分析表.groupby(["月份","渠道"]).agg(纳入订单数=("订单编号","nunique"),
                                  已知金额合计=("实付金额","sum"),
                           平均订单金额=("实付金额","mean"))
渠道月报

In [ ]:
渠道对比 = 渠道月报.reset_index().pivot(
    index = "渠道",
    columns = "月份",
    values = "已知金额合计")
渠道对比

In [ ]:
ax = 渠道对比.plot.bar(
    figsize = (7,4),
    rot = 0
)
ax.set_title("各渠道已知金额对比（仅日期和金额完整的订单）")
ax.set_xlabel("渠道")
ax.set_ylabel("实付金额合计（元）")

plt.tight_layout()
plt.show()

## 官网各商品的变化

In [ ]:
官网订单 = 分析表.loc[分析表["渠道"] == "官网"].copy()

In [ ]:
商品月报 = 官网订单.groupby(["月份","商品编号"]).agg(订单数=("订单编号","nunique"),
    售出件数=("数量","sum"),
    实付金额合计=("实付金额","sum")
)
商品月报

In [ ]:
商品对比 = 商品月报.reset_index().pivot(
    index = "商品编号",
    columns = "月份",
    values = "实付金额合计"
)
商品对比

In [ ]:
商品对比 = 商品对比.fillna(0)

In [ ]:
商品对比["金额变化"] = (商品对比[pd.Period("2025-07",freq = "M")]
- 商品对比[pd.Period("2025-06",freq = "M")]
               )
商品对比

In [ ]:
P03订单 = 分析表.loc[分析表["商品编号"] == "P03"].copy()
P03订单

In [ ]:
P03平台对比 = P03订单.groupby(["月份","渠道"]).agg(订单数=("订单编号","nunique"),
                                         售出件数=("数量","sum"),
                                               实付金额合计=("实付金额","sum"))
P03平台对比

## 关联商品目录

In [ ]:
商品资料 = pd.DataFrame({
    "商品编号": ["P01", "P02", "P03"],
    "商品名称": ["马克杯", "保温杯", "咖啡壶"],
    "品类": ["饮具", "饮具", "咖啡器具"]
})

商品资料

In [ ]:
订单详情 = 分析表.merge(
    商品资料,
    on="商品编号",
    how="left",
    validate="many_to_one"
)
订单详情

In [ ]:
print("关联前行数：",len(分析表))
print("关联后行数：",len(订单详情))
订单详情.loc[订单详情["商品名称"].isna(),
    ["订单编号","商品编号"]
    ]

## 分析结论

本分析使用模拟订单数据，已处理重复记录、渠道空格，
并根据教学补查信息补齐金额和日期。
假设补查后的数据完整覆盖六月、七月。

- 七月销售额由六月的 2600 元降至 1600 元。
- 官网减少 1000 元，平台总额保持 950 元不变。
- 官网内部，咖啡壶（P03）减少 900 元，
  马克杯（P01）减少 400 元，保温杯（P02）增加 300 元。
- 咖啡壶在平台的金额由 300 元增至 600 元，
  但现有数据无法证明用户发生了跨渠道转移。

## 后续调查

优先核查咖啡壶在官网七月的上架状态、库存和访问数据。
当前订单数据能够定位变化，尚不能确定下降原因。